# Outflow boundaries & acoustic power: choked nozzle and metered bleed

Beyond the static-pressure outlet, FNS has two flow-fixing outflow boundaries:

* **`mass_flow_outlet`** — pins the outflow rate; the static pressure floats. Its natural
  acoustic closure is **constant mass flow**, $\dot m' = 0$.
* **`choked_nozzle_outlet`** — a compact sonic throat of area $A^\*$ lumped just downstream.
  The outflow is the **critical mass flux** for the interior stagnation state, the approach
  plane stays subsonic, and its acoustic closure is the compact choked-nozzle
  (**Marble–Candel**) reflection, entropy coupling included.

A key property of FNS: each boundary's acoustic closure is the *linearization of its mean
residual* — so `mass_flow_outlet` inherits $\dot m'=0$ and `choked_nozzle_outlet` inherits
Marble–Candel **with no separately specified boundary condition**. Below we don't just claim
that — we pull the closure straight out of the converged Jacobian $J_\text{alg}$ and check it
against the textbook formulas.

We then use the new **acoustic-power diagnostics** (`fns.perturbation.power`) to read the
energy budget of the network: which boundaries feed acoustic energy and which absorb it. This
turns out to be essential, because with mean flow a reflection coefficient $|R|>1$ does **not**
mean energy is being created, and a boundary specified as merely "partially reflecting" can
secretly be an acoustic *source*.

In [ ]:
import os, sys, warnings

_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "fns")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np
import plotly.graph_objects as go

from fns.elements import catalog as cat
from fns.elements.ids import CHOKED_NOZZLE_OUTLET, MASS_FLOW_OUTLET
from fns.perturbation.boundary_bc import PerturbationBC
from fns.perturbation import eigenmodes, find_terminals
from fns.perturbation import acoustic_intensity, acoustic_energy_density, passive_reflection_bound
from fns.perturbation.operator import build_acoustic_blocks
from fns.perturbation.characteristics import char_to_dx
from fns.plotting import use_fns_theme
from fns.solver import solve
from fns.solver.control import states_table
from fns.thermo.configure import perfect_gas
from fns.derive import ES_MDOT, ES_M, ES_P, ES_RHO, ES_C, ES_U, ES_AREA

use_fns_theme()
R, GAMMA = 287.0, 1.4
CP = GAMMA * R / (GAMMA - 1.0)
K = CP / R

## 1. A branched rig: choked core nozzle + metered bleed

A reservoir (total-pressure inlet) feeds a duct into a **splitter**; one branch exhausts
through a **choked nozzle** (throat $A^\*=0.015\,\mathrm{m^2}$), the other is a **metered
bleed** drawing a fixed $2\,\mathrm{kg/s}$:

```
reservoir(pt) ── feed ──[splitter]── core ── (choked nozzle A*)
                              └──── bleed ── (mass-flow outlet)
```

The reservoir pins the pressure level; the choke and the bleed each pin a branch flow, so
the inlet mass flow self-adjusts to supply both. Everything stays subsonic — the sonic
point lives inside the lumped nozzle throat, not in the domain.

In [ ]:
def build_rig(inlet_R=0.8):
    els = [
        cat.total_pressure_inlet(2.5e5, 300.0, name="reservoir", perturbation_bc=PerturbationBC.reflection(inlet_R)),
        cat.duct(0.6, name="feed"),
        cat.splitter(name="manifold"),
        cat.duct(0.4, name="core"),
        cat.choked_nozzle_outlet(0.015, name="nozzle"),   # inherits Marble-Candel
        cat.duct(0.5, name="bleedpipe"),
        cat.mass_flow_outlet(2.0, name="bleed"),           # inherits mdot' = 0
    ]
    edges = [(0, 1, 0.05), (1, 2, 0.05), (2, 3, 0.03), (3, 4, 0.03), (2, 5, 0.02), (5, 6, 0.02)]
    return cat.build_problem(perfect_gas(R, GAMMA), els, edges, mdot_ref=6.0, p_ref=1.5e5, h_ref=CP * 300.0)

prob = build_rig()
res = solve(prob)
assert res.converged
est = states_table(prob, res.x)
print(f"reservoir feed : mdot = {est[ES_MDOT,1]:6.3f} kg/s   M = {est[ES_M,1]:.3f}   p = {est[ES_P,1]:8.0f} Pa")
print(f"choked core    : mdot = {est[ES_MDOT,3]:6.3f} kg/s   M = {est[ES_M,3]:.3f}   (sonic-throat critical flux)")
print(f"metered bleed  : mdot = {est[ES_MDOT,5]:6.3f} kg/s   M = {est[ES_M,5]:.3f}   (pinned)")
print(f"mass balance   : feed {est[ES_MDOT,1]:.3f}  =  nozzle {est[ES_MDOT,3]:.3f} + bleed {est[ES_MDOT,5]:.3f}")

## 2. Well-posedness: every case needs a pressure reference

The new flow-fixing outlets make it easy to write an **ill-posed** boundary set. If *every*
boundary only pins a mass flow (`mass_flow_inlet`, `mass_flow_outlet`, `wall`), the absolute
pressure level is a free gauge — adding a constant to every pressure leaves the residual
unchanged — so the steady Jacobian is singular. FNS catches this **before solving**:

In [ ]:
def try_build(elements, edges):
    try:
        cat.build_problem(perfect_gas(R, GAMMA), elements, edges, mdot_ref=5.0, p_ref=1e5, h_ref=CP * 300.0)
        return "built OK"
    except ValueError as e:
        return f"REJECTED: {e}"

edges2 = [(0, 1, 0.05), (1, 2, 0.05)]
print("mass_flow_inlet + mass_flow_outlet:")
print("  ", try_build([cat.mass_flow_inlet(5.0, 300.0), cat.duct(0.4), cat.mass_flow_outlet(5.0)], edges2))
print("\nmass_flow_inlet + choked_nozzle_outlet:")
print("  ", try_build([cat.mass_flow_inlet(5.0, 300.0), cat.duct(0.4), cat.choked_nozzle_outlet(0.03)], edges2))

A `total_pressure_inlet`/`pressure_outlet` pins the level directly; a
`choked_nozzle_outlet` pins it through its critical-mass-flux relation. So
`mass_flow_inlet + choked_nozzle_outlet` is fine.

## 3. The acoustic closures come straight out of $J_\text{alg}$

The claim "inherited for free" is concrete: the converged complex-step Jacobian $J_\text{alg}$
is exactly the zero-frequency perturbation operator, so the outlet's *one row* in
$J_\text{alg}$, rotated into characteristic coordinates $(f,g,h)$, **is** the acoustic closure.
Reading $c_f f + c_g g + c_h h = 0$ at the outlet gives $g = R\,f + R_s\,h$ with $R=-c_f/c_g$,
$R_s=-c_h/c_g$. We compare those numbers against the textbook formulas — no hand-coded BC was
applied anywhere.

In [ ]:
blocks = build_acoustic_blocks(prob, res.x)
ns = int(prob.n_solve)
terms = {t.rid: t for t in find_terminals(prob, res.x)}

def inherited_closure(term):
    """(R, R_s) read from the boundary's J_alg row, in characteristic coordinates."""
    e = term.edge
    r0 = int(prob.node_row_ptr[term.node])
    row = np.asarray(blocks.J_alg[r0, ns * e : ns * e + 3].todense()).ravel()
    state = [float(est[k, e]) for k in (ES_RHO, ES_C, ES_U, ES_P, ES_AREA)]
    cf, cg, ch = row @ char_to_dx(*state, K)
    return (-cf / cg, -ch / cg)  # outlet: g = R f + R_s h

# choked nozzle  ->  Marble-Candel
tn = terms[CHOKED_NOZZLE_OUTLET]
Mn = float(est[ES_M, tn.edge]); rho_n, c_n = float(est[ES_RHO, tn.edge]), float(est[ES_C, tn.edge])
R_inh, Rs_inh = inherited_closure(tn)
R_mc = (2 - (GAMMA - 1) * Mn) / (2 + (GAMMA - 1) * Mn)
Rs_mc = (c_n / rho_n) * Mn / (2 + (GAMMA - 1) * Mn)
print(f"choked nozzle (M={Mn:.3f}):")
print(f"   R   : J_alg {R_inh.real:+.6f}   Marble-Candel {R_mc:+.6f}   |diff| {abs(R_inh - R_mc):.1e}")
print(f"   R_s : J_alg {Rs_inh.real:+.4f}   Marble-Candel {Rs_mc:+.4f}   |diff| {abs(Rs_inh - Rs_mc):.1e}")

# mass-flow outlet  ->  constant mass flow
tb = terms[MASS_FLOW_OUTLET]
Mb = float(est[ES_M, tb.edge])
R_inh_b, _ = inherited_closure(tb)
R_cmf = (1 + Mb) / (1 - Mb)
print(f"\nmass-flow outlet (M={Mb:.3f}):")
print(f"   R   : J_alg {R_inh_b.real:+.6f}   (1+M)/(1-M) {R_cmf:+.6f}   |diff| {abs(R_inh_b - R_cmf):.1e}")

So both closures fall out of the mean-flow Jacobian to round-off. Now the two reflection
laws across approach Mach. Note where they sit relative to $R=1$ — and hold that thought:
$R\to1$ for both as $M\to0$ (a slow approach looks rigid), the choked nozzle drops *below* 1,
and constant mass flow climbs *above* 1.

In [ ]:
M = np.linspace(0.0, 0.6, 121)
rho, c = 1.2, 340.0
R_choked = [PerturbationBC.choked_nozzle().reflection_coefficient(0.0, rho, c, m, K).real for m in M]
R_cmf = [PerturbationBC.constant_mass_flow().reflection_coefficient(0.0, rho, c, m).real for m in M]
fig = go.Figure()
fig.add_scatter(x=M, y=R_choked, mode="lines", name="choked nozzle (Marble-Candel)")
fig.add_scatter(x=M, y=R_cmf, mode="lines", name="constant mass flow  (mdot' = 0)")
fig.add_hline(y=1.0, line_dash="dot", annotation_text="rigid wall R = +1")
fig.update_layout(title="Inherited outlet reflection vs. approach Mach",
                  xaxis_title="approach Mach M", yaxis_title="reflection coefficient R", template="fns")
fig.show()

## 4. Acoustic power with mean flow: why $|R|>1$ is *not* energy growth

With a moving mean flow, acoustic energy is **not** carried in proportion to $|p'|^2$. The
time-averaged energy flux of the down/up-running characteristics $f,g$ across a section is

$$ I \;=\; \tfrac12\rho c\big[(1+M)^2|f|^2 \;-\; (1-M)^2|g|^2\big]. $$

The mean flow biases the two directions by $(1\pm M)^2$: a downstream wave is a *stronger*
energy carrier than an upstream one of equal amplitude. Two consequences settle the puzzles:

* **Constant mass flow, $R=\frac{1+M}{1-M}>1$, is lossless.** The reflected (upstream) wave
  needs amplitude $\frac{1+M}{1-M}$ just to carry back the *same power* it received. Reflected
  power $/$ incident power $= R^2\frac{(1-M)^2}{(1+M)^2}=1$ exactly — a perfect mirror, no
  energy made. The amplitude exceeds 1 only because of the $(1\pm M)^2$ bookkeeping.
* **Choked nozzle, $R<1$, absorbs.** Its $R^2\frac{(1-M)^2}{(1+M)^2}<1$: most of the incident
  acoustic power *leaves through the throat* (convected out past the sonic point — it is not a
  closed end). At the rig's $M\approx0.31$ only ~20% is reflected.

The energy-neutral (lossless) reflection magnitude is therefore $\frac{1+M}{1-M}$ at an outlet
and $\frac{1-M}{1+M}$ at an inlet — anything *above* that bound is an acoustic source. We plot
those bounds with `passive_reflection_bound`:

In [ ]:
# the primitives in action: a unit f-wave's energy travels at u+c, a unit g-wave at u-c
m0, rho0, c0 = 0.3, 1.2, 340.0
v_fwd = acoustic_intensity(rho0, c0, m0, 1.0, 0.0) / acoustic_energy_density(rho0, m0, 1.0, 0.0)
v_bwd = acoustic_intensity(rho0, c0, m0, 0.0, 1.0) / acoustic_energy_density(rho0, m0, 0.0, 1.0)
print(f"energy group speed @ M={m0}:  forward {v_fwd:.1f} = u+c {c0*(1+m0):.1f}   "
      f"backward {v_bwd:.1f} = u-c {c0*(m0-1):.1f}")

fig = go.Figure()
neutral_out = [passive_reflection_bound(m, "outlet") for m in M]
neutral_in = [passive_reflection_bound(m, "inlet") for m in M]
fig.add_scatter(x=M, y=neutral_out, mode="lines", line=dict(color="#d62728", dash="dash"),
                name="outlet neutral |R| = (1+M)/(1-M)")
fig.add_scatter(x=M, y=R_choked, mode="lines", name="choked nozzle  (absorbs: below neutral)")
fig.add_scatter(x=M, y=R_cmf, mode="lines", line=dict(color="#2ca02c"),
                name="constant mass flow  (on neutral: lossless)")
fig.add_scatter(x=M, y=neutral_in, mode="lines", line=dict(color="#1f77b4", dash="dash"),
                name="inlet neutral |R| = (1-M)/(1+M)")
fig.add_hline(y=0.8, line_dash="dot", annotation_text="specified inlet |R| = 0.8")
fig.add_hline(y=1.0, line_dash="dot", line_color="#aaa")
fig.update_layout(
    title="Energy-neutral reflection bounds: |R| above the bound = acoustic source",
    xaxis_title="approach Mach M", yaxis_title="|R|", yaxis_range=[0, 2.2], template="fns")
fig.show()
Min = float(est[ES_M, 1])
print(f"At the rig's inlet M = {Min:.3f}, the energy-neutral inlet |R| is only "
      f"{passive_reflection_bound(Min, 'inlet'):.3f}.")
print("The specified inlet |R| = 0.8 is ABOVE that -> the 'partially reflecting' inlet is an acoustic SOURCE.")

## 5. The spectrum, and *attributing* it with the boundary power budget

There is no heat source in this rig, so a growing acoustic mode should be impossible for a
*passive* network. Yet with the inlet at $|R|=0.8$ the spectrum shows a growing mode. The
acoustic-power diagnostic explains exactly why: for a self-sustained mode the global energy
balance is $dE/dt = 2\sigma E = \sum_\text{boundaries} W_\text{in}$, so the **sign of the net
boundary power equals the sign of the growth rate**, and the per-boundary breakdown tells us
*who* drives it. `result.boundary_power(i)` computes the Myers flux through each terminal.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    spec = eigenmodes(prob, res.x, freq_band=(50.0, 500.0), growth_band=(-150.0, 150.0), isentropic=True)
for i in range(spec.n_modes):
    m = spec.summary()[i]
    flag = "  <-- growing" if m["unstable"] else ""
    print(f"mode {i}:  f = {m['freq_hz']:7.2f} Hz    growth = {m['growth_rate']:+8.2f} 1/s{flag}")

i_grow = int(np.argmax(spec.growth_rates))
bp = spec.boundary_power(i_grow)
print()
print(bp.table())
spec.plot_spectrum(title="Branched rig: acoustic spectrum (inlet |R| = 0.8)").show()

In [ ]:
bp.plot(title=f"Who drives the {bp.freq_hz:.0f} Hz mode? (red = source, blue = sink)").show()

The bar chart attributes the growth: the **reservoir inlet is the source** (it reflects
at $|R|=0.8$, above its energy-neutral $0.64$), the **choked nozzle is the sink** (it lets
acoustic power escape through the throat), and the **mass-flow bleed is exactly neutral**
(a lossless mirror, $0\%$). The "instability" is an artifact of an over-reflecting inlet
boundary, not a physical mechanism.

Drop the inlet below its passivity bound and the network is unconditionally dissipative — the
net boundary power flips negative and every mode decays:

In [ ]:
prob_p = build_rig(inlet_R=0.5)   # 0.5 < (1-M)/(1+M) = 0.64  ->  passive inlet
res_p = solve(prob_p)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    spec_p = eigenmodes(prob_p, res_p.x, freq_band=(50.0, 500.0), growth_band=(-150.0, 150.0), isentropic=True)
for i in range(spec_p.n_modes):
    bp_p = spec_p.boundary_power(i)
    print(f"f = {bp_p.freq_hz:7.2f} Hz   growth = {bp_p.growth_rate:+7.2f} 1/s   "
          f"net boundary power {100*bp_p.net/bp_p.gross:+6.1f}%   "
          f"{'sign OK' if bp_p.sign_consistent else 'SIGN MISMATCH'}")

## Summary

* `mass_flow_outlet` (pinned outflow, floating pressure) and `choked_nozzle_outlet` (compact
  sonic throat, critical mass flux) join `pressure_outlet` as **outflow-only** boundaries.
* Their acoustic closures — constant mass flow $\dot m'=0$ and Marble–Candel — are **read out
  of $J_\text{alg}$** to round-off (§3); no separate BC is specified.
* The build-time check rejects boundary sets with **no pressure reference** (§2).
* **Acoustic-power diagnostics** (`fns.perturbation.power`): with mean flow the energy-neutral
  reflection magnitude is $\frac{1+M}{1-M}$ (outlet) / $\frac{1-M}{1+M}$ (inlet), so $|R|>1$ can
  be lossless and a "partially reflecting" inlet can be a source. `result.boundary_power(i)`
  gives the per-boundary energy budget, whose net shares a sign with the growth rate
  ($2\sigma E = \sum W_\text{in}$) and attributes any instability to the boundary feeding it.